# Network Traffic Security Analysis (UNSW-NB15)

## Objective
The goal of this project is to analyze network traffic data and identify patterns related to cyber attacks.

## Dataset
UNSW-NB15 dataset containing normal and malicious network traffic.

## Tools Used
- Python (pandas, matplotlib, seaborn)
- Scikit-learn (RandomForest)

In [ ]:

# =========================================
# Network Traffic Analysis (UNSW-NB15)
# =========================================

# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style("whitegrid")

#Load datasets

# Update the file paths according to your local setup
train_path = r'C:\Users\avinr\OneDrive\Desktop\Cyber_Security_Analysis\UNSW_NB15_training-set.csv'
test_path = r'C:\Users\avinr\OneDrive\Desktop\Cyber_Security_Analysis\UNSW_NB15_testing-set.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

#Dataset overview
# -----------------------------------------

print("Training dataset shape:", train.shape)
print("Testing dataset shape:", test.shape)

#Preview the training dataset
# -----------------------------------------

train.head()

## Attack Category Distribution

This section analyzes the distribution of different attack types to understand which attacks are most common in the dataset.

In [ ]:
# =========================================
#Attack Category Distribution Analysis
# =========================================

# Create a new figure with specified size (width=12, height=6)
plt.figure(figsize=(12, 6))

# Plot the distribution of attack categories
sns.countplot(
    data=train,
    y='attack_cat',
    order=train['attack_cat'].value_counts().index
)

# Add titles and labels
plt.title('Distribution of Attack Categories')
plt.xlabel('Number of Records')
plt.ylabel('Attack Type')

# Display the plot
plt.show()

# -----------------------------------------
# Print frequency of each attack category
# -----------------------------------------

print(train['attack_cat'].value_counts())

In [ ]:
# =========================================
# Missing Values & Data Types
# =========================================

# Check for missing values
print("Total Missing Values:")
print(train.isnull().sum().sum())

#Check column data types

print("\nData Types Summary:")
print(train.dtypes.value_counts())


## Protocol Analysis

This section examines the relationship between network protocols and attack types to identify suspicious patterns.

In [ ]:
# =========================================
# Protocol-Based Normal vs Attack Analysis
# =========================================

# Select top 5 most frequent protocols
top_protos = train['proto'].value_counts().nlargest(5).index

# Create a figure
plt.figure(figsize=(10, 6))

# Filter dataset to include only top protocols
filtered_data = train[train['proto'].isin(top_protos)]

# Plot distribution of Normal (0) vs Attack (1) traffic by protocol
sns.countplot(
    data=filtered_data,
    x='proto',
    hue='label'
)

# Add title and legend
plt.title('Protocol-Based Distribution of Normal (0) vs Attack (1) Traffic')
plt.xlabel('Protocol')
plt.ylabel('Count')
plt.legend(title='Status', labels=['Normal', 'Attack'])

# Display the plot
plt.show()

## Data Visualization

Visualizations are used to better understand patterns, anomalies, and relationships in the data.

In [ ]:
# =========================================
# Most Targeted Services (Attack Traffic Only)
# =========================================

# Filter only attack traffic (label = 1)
attacks = train[train['label'] == 1]

# Create a figure
plt.figure(figsize=(12, 6))

# Plot most targeted services based on attack traffic
sns.countplot(
    data=attacks,
    x='service',
    order=attacks['service'].value_counts().index
)

# Add title and labels
plt.title('Most Targeted Services (Attack Traffic Only)')
plt.xlabel('Service')
plt.ylabel('Number of Attacks')

# Rotate x labels for better readability (important!)
plt.xticks(rotation=45)

# Display the plot
plt.show()

# Top 5 most targeted services
print("Top 5 Most Targeted Services:")
print(attacks['service'].value_counts().head(5))

In [ ]:
# =========================================
# Source Packets vs Source Bytes
# =========================================

# Create a figure
plt.figure(figsize=(8, 7))

# Plot scatterplot with a sample of data (to improve performance)
sns.scatterplot(
    data=train.sample(2000),
    x='spkts',
    y='sbytes',
    hue='label',
    alpha=0.5
)

# Add title and labels
plt.title('Source Packets vs Source Bytes (Normal=0, Attack=1)')
plt.xlabel('Source Packets')
plt.ylabel('Source Bytes')

# Display the plot
plt.show()

## Machine Learning Model

A RandomForest classifier is used to classify network traffic as normal or attack based on selected features.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import numpy as np

In [ ]:
# Create copies of training and testing datasets to avoid modifying original data
train_ml = train.copy()
test_ml = test.copy()

# Convert categorical (text) columns into numerical values
le = LabelEncoder()
cat_cols = ['proto', 'service', 'state']

for col in cat_cols:
    # Fit and transform training data
    train_ml[col] = le.fit_transform(train_ml[col])
    
    # Handle unseen categories in test data by assigning '<unknown>'
    test_ml[col] = test_ml[col].map(lambda s: '<unknown>' if s not in le.classes_ else s)
    
    # Add '<unknown>' to known classes
    le.classes_ = np.append(le.classes_, '<unknown>')
    
    # Transform test data
    test_ml[col] = le.transform(test_ml[col])

# Drop unnecessary columns (ID and attack category are not useful for the model)
X_train = train_ml.drop(['id', 'label', 'attack_cat'], axis=1)
y_train = train_ml['label']

X_test = test_ml.drop(['id', 'label', 'attack_cat'], axis=1)
y_test = test_ml['label']

In [ ]:
model=RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("ok")


## Model Results

The model is evaluated based on its ability to correctly classify normal and malicious traffic.

In [ ]:
# Make predictions on the test dataset
y_pred = model.predict(X_test)

# Print model accuracy
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## Key Findings

- Certain attack types appear more frequently in the dataset
- Specific protocols are associated with higher attack activity
- Patterns in network traffic can indicate potential threats

## Conclusion

This project demonstrates how data analysis and machine learning can be applied in cybersecurity to detect malicious activity.